# MaskSDM Training & Benchmarking Pipeline
Drop-in replacement for the CISO notebook. Uses identical data, splits,
species filter (>=100 occurrences), and STEM-LM metric set.
Only evaluates p=1.0 (100% masking / fully unconditioned).

**Steps:**
1. Setup
2. Load and inspect data  (identical to CISO notebook)
3. Build data arrays      (no file reformatting needed)
4. Verify integrity
5. Build config and train
6. Inference with 100% masking
7. STEM-LM metrics

## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import subprocess

# ── Paths — keep DATA_DIR identical to CISO notebook so the same files are used
MASKSDM_DIR = '/content/MaskSDM-MEE'
DATA_DIR    = '/content/drive/MyDrive/CISO/data'
CKPT_DIR    = '/content/drive/MyDrive/MaskSDM/model_checkpoints_1339'
RESULTS_DIR = '/content/drive/MyDrive/MaskSDM/results_plants_1000_epochs_1339'

for d in [DATA_DIR, CKPT_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

# Clone MaskSDM repo to local Colab storage
if not os.path.exists(MASKSDM_DIR):
    subprocess.run(
        ['git', 'clone', 'https://github.com/zbirobin/MaskSDM-MEE', MASKSDM_DIR],
        check=True
    )
    print('Repo cloned.')
else:
    print('Repo already cloned, skipping.')

os.chdir(MASKSDM_DIR)
print('Working directory:', os.getcwd())
print('\nSetup complete.')

Mounted at /content/drive
Repo cloned.
Working directory: /content/MaskSDM-MEE

Setup complete.


In [ ]:
!pip install verde schedulefree elapid torcheval -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.6/188.6 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.1/55.1 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 436.5/436.5 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.2/179.2 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 110.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
umap-learn 0.5.12 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
hdbscan 0.8.42 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.


In [ ]:
import sys
sys.path.insert(0, MASKSDM_DIR)

import numpy as np
import pandas as pd
import json
import torch
from pathlib import Path
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score, average_precision_score
from scipy.stats import spearmanr

from data_helpers import get_torch_dataset
from modules import get_model
from training_helpers import seed_everything, train

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print('Imports OK')

Device: cuda
Imports OK


## 1. Load Your Data

In [ ]:
# ── Identical to CISO notebook ────────────────────────────────────────────────
DATA_CSV    = f'{DATA_DIR}/splotopen_global.csv'
SPLITS_JSON = f'{DATA_DIR}/splotopen_global_splits.json'

for f in [DATA_CSV, SPLITS_JSON]:
    print('OK  ' if os.path.exists(f) else 'MISSING  ', f)

plants = pd.read_csv(DATA_CSV)
plants = plants.reset_index(drop=True)

with open(SPLITS_JSON) as f:
    plants_splits = json.load(f)

print(f'\nLoaded {len(plants):,} rows x {len(plants.columns):,} columns')
print(f'Split keys: {list(plants_splits.keys())}')
plants.head(2)

OK   /content/drive/MyDrive/CISO/data/splotopen_global.csv
OK   /content/drive/MyDrive/CISO/data/splotopen_global_splits.json

Loaded 95,104 rows x 1,232 columns
Split keys: ['num_rows', 'meta', 'train', 'val', 'test']


,time,latitude,longitude,env_bio01,env_bio02,env_bio03,env_bio04,env_bio05,env_bio06,env_bio07,...,Erigeron uniflorus,Impatiens meruensis,Geranium sanguineum,Chamaerhodos erecta,Geranium pratense,Cassytha filiformis,Rytidosperma setaceum,Serenoa repens,Hydrocotyle vulgaris,Phillyrea angustifolia
0,0.0,62.42,-154.18,-2.608333,10.45,25.059952,1163.3374,18.5,-23.2,41.7,...,0,0,0,0,0,0,0,0,0,0
1,0.0,62.42,-154.18,-2.608333,10.45,25.059952,1163.3374,18.5,-23.2,41.7,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
# ── Identical column identification to CISO notebook ─────────────────────────
env_cols     = [c for c in plants.columns if c.startswith('env_')]
species_cols = [c for c in plants.columns
                if c not in env_cols + ['time', 'latitude', 'longitude']]

worldclim_cols = [c for c in env_cols if 'bio' in c.lower()]
soilgrid_cols  = [c for c in env_cols if 'bio' not in c.lower()]

print(f'WorldClim cols ({len(worldclim_cols)}): {worldclim_cols}')
print(f'SoilGrid cols  ({len(soilgrid_cols)}):  {soilgrid_cols}')
print(f'Species cols   ({len(species_cols)}):   first 5 = {species_cols[:5]}')

WorldClim cols (19): ['env_bio01', 'env_bio02', 'env_bio03', 'env_bio04', 'env_bio05', 'env_bio06', 'env_bio07', 'env_bio08', 'env_bio09', 'env_bio10', 'env_bio11', 'env_bio12', 'env_bio13', 'env_bio14', 'env_bio15', 'env_bio16', 'env_bio17', 'env_bio18', 'env_bio19']
SoilGrid cols  (9):  ['env_bdod', 'env_cec', 'env_cfvo', 'env_clay', 'env_nitrogen', 'env_phh2o', 'env_sand', 'env_silt', 'env_dem']
Species cols   (1201):   first 5 = ['Vaccinium myrtillus', 'Deschampsia flexuosa', 'Anthoxanthum odoratum', 'Festuca rubra', 'Achillea millefolium']


## 2. Build Split Indices and Data Arrays

In [ ]:
# ── Identical split logic to CISO notebook ────────────────────────────────────
train_split = np.array(plants_splits['train'])
val_split   = np.array(plants_splits['val'])
test_split  = np.array(plants_splits['test'])

assert train_split.max() < len(plants), 'Train index out of bounds'
assert val_split.max()   < len(plants), 'Val index out of bounds'
assert test_split.max()  < len(plants), 'Test index out of bounds'
print(f'train={len(train_split):,}  val={len(val_split):,}  test={len(test_split):,}')
print('All indices in bounds.')

train=76,699  val=8,636  test=9,769
All indices in bounds.


In [ ]:
# ── Filter species: >=100 occurrences (identical threshold to CISO notebook) ─
targets_full   = plants[species_cols].to_numpy().astype(np.float32)
species_counts = targets_full.sum(axis=0)
keep           = species_counts >= 100
targets        = targets_full[:, keep]
species_cols_filtered = [s for s, k in zip(species_cols, keep) if k]

print(f'Species before filtering: {len(species_cols):,}')
print(f'Species after  filtering: {len(species_cols_filtered):,}')

# ── Tabular env features ──────────────────────────────────────────────────────
tabular_x = plants[env_cols].to_numpy().astype(np.float32)

# ── SatCLIP embeddings: not available, use zeros (masked out during training) ─
# MaskSDM's extra_masking will mask these just like real predictors.
N_SATCLIP = 256
satclip_embeddings = np.zeros((len(plants), N_SATCLIP), dtype=np.float32)

# ── Build data dict in MaskSDM format ────────────────────────────────────────
data = {
    'tabular_x':          tabular_x,
    'y':                  targets,
    'satclip_embeddings': satclip_embeddings,
}

# ── Per-split arrays ──────────────────────────────────────────────────────────
data['x_train'] = tabular_x[train_split]
data['x_val']   = tabular_x[val_split]
data['x_test']  = tabular_x[test_split]
data['y_train'] = targets[train_split]
data['y_val']   = targets[val_split]
data['y_test']  = targets[test_split]
data['satclip_embeddings_train'] = satclip_embeddings[train_split]
data['satclip_embeddings_val']   = satclip_embeddings[val_split]
data['satclip_embeddings_test']  = satclip_embeddings[test_split]

# ── Normalise using train statistics only ─────────────────────────────────────
train_mean = np.nanmean(data['x_train'], axis=0)
train_std  = np.nanstd(data['x_train'],  axis=0)
data['x_train'] = (data['x_train'] - train_mean) / (train_std + 1e-4)
data['x_val']   = (data['x_val']   - train_mean) / (train_std + 1e-4)
data['x_test']  = (data['x_test']  - train_mean) / (train_std + 1e-4)

print(f'x_train: {data["x_train"].shape}')
print(f'x_val:   {data["x_val"].shape}')
print(f'x_test:  {data["x_test"].shape}')

Species before filtering: 1,201
Species after  filtering: 1,201
x_train: (76699, 28)
x_val:   (8636, 28)
x_test:  (9769, 28)


## 3. Verify Integrity

In [ ]:
n_features = data['x_train'].shape[1]
n_species  = data['y_train'].shape[1]

# Species that appear in all three splits — used for evaluation
indices_evaluated = np.intersect1d(
    np.intersect1d(
        data['y_train'].sum(axis=0).nonzero()[0],
        data['y_val'].sum(axis=0).nonzero()[0]
    ),
    data['y_test'].sum(axis=0).nonzero()[0]
).tolist()

print(f'n_features:        {n_features}')
print(f'n_species:         {n_species}')
print(f'species_evaluated: {len(indices_evaluated)}')
print(f'train rows:        {len(data["x_train"]):,}')
print(f'val rows:          {len(data["x_val"]):,}')
print(f'test rows:         {len(data["x_test"]):,}')
print('\nAll checks passed.')

n_features:        28
n_species:         1201
species_evaluated: 950
train rows:        76,699
val rows:          8,636
test rows:         9,769

All checks passed.


## 4. Build Config and Train

In [ ]:
random_seed = 1339  # match CISO notebook seed
seed_everything(random_seed)
torch.set_default_device(device)

# ── Edit these ────────────────────────────────────────────────────────────────
MAX_EPOCHS = 1000     # match CISO notebook
BATCH_SIZE = 256
SAVE_DIR   = f'{CKPT_DIR}/masksdm_splot'
# ─────────────────────────────────────────────────────────────────────────────

config = {
    'device':                    device,
    'seed':                      random_seed,
    'dataset':                   'splot',
    'n_features':                n_features,
    'n_species':                 n_species,
    'n_samples_train':           len(data['y_train']),
    'n_samples_val':             len(data['y_val']),
    'n_samples_test':            len(data['y_test']),
    'indices_evaluated_species': indices_evaluated,
    'n_evaluated_species':       len(indices_evaluated),
    # satclip=False: zeros are always fully masked so they carry no signal
    'satclip':                   False,
    'model':                     'FTTransformer',
    'd_hidden':                  192,
    'n_heads':                   8,
    'n_blocks':                  7,
    'n_layers':                  7,
    'dropout':                   0.1,
    'd_out':                     n_species,
    'epochs':                    MAX_EPOCHS,
    'batch_size':                BATCH_SIZE,
    'batch_size_eval':           4096,
    'loss':                      'weighted',
    'species_weights':           torch.tensor(
                                     len(data['y_train']) / (data['y_train'].sum(axis=0) + 1e-5),
                                     dtype=torch.float32
                                 ).to(device),
    'optimizer':                 'AdamW',
    'scheduler_free':            True,
    'lr':                        0.001,
    'weight_decay':              0.01,
    'warmup_steps':              1000,
    'masking':                   True,
    'extra_masking':             True,
    'save_dir':                  SAVE_DIR,
    'use_wandb':                 False,
    'wandb_init':                {},
}

print(f'n_features:  {n_features}')
print(f'n_species:   {n_species}')
print(f'max_epochs:  {MAX_EPOCHS}')
print(f'save_dir:    {SAVE_DIR}')
print(f'device:      {device}')

n_features:  28
n_species:   1201
max_epochs:  1000
save_dir:    /content/drive/MyDrive/MaskSDM/model_checkpoints_1339/masksdm_splot
device:      cuda


In [ ]:
train(config, data)

Training model...
Epoch 1, val AUC: 0.741298
Epoch 2, val AUC: 0.854294
Epoch 3, val AUC: 0.874312
Epoch 4, val AUC: 0.889325
Epoch 5, val AUC: 0.894838
Epoch 6, val AUC: 0.899577
Epoch 7, val AUC: 0.901097
Epoch 8, val AUC: 0.902787
Epoch 9, val AUC: 0.905067
Epoch 10, val AUC: 0.907132
Epoch 11, val AUC: 0.908238
Epoch 12, val AUC: 0.909654
Epoch 13, val AUC: 0.910408
Epoch 14, val AUC: 0.910547
Epoch 15, val AUC: 0.911487
Epoch 16, val AUC: 0.912520
Epoch 17, val AUC: 0.913496
Epoch 18, val AUC: 0.913888
Epoch 19, val AUC: 0.914515
Epoch 20, val AUC: 0.915434
Epoch 21, val AUC: 0.915911
Epoch 22, val AUC: 0.916111
Epoch 23, val AUC: 0.915975
Epoch 24, val AUC: 0.915941
Epoch 25, val AUC: 0.915888
Epoch 26, val AUC: 0.915827
Epoch 27, val AUC: 0.915623
Epoch 28, val AUC: 0.915610
Epoch 29, val AUC: 0.915607
Epoch 30, val AUC: 0.916067
Epoch 31, val AUC: 0.916384
Epoch 32, val AUC: 0.916676
Epoch 33, val AUC: 0.916736
Epoch 34, val AUC: 0.916891
Epoch 35, val AUC: 0.916915
Epoch 36, v

FTTransformer(
  (feature_tokenizer): FeatureTokenizer(
    (periodic_embeddings): PeriodicEmbeddings(
      (periodic): _Periodic()
      (linear): _NLinear()
      (activation): ReLU()
    )
  )
  (satclip_projection): Linear(in_features=256, out_features=192, bias=True)
  (transformer_encoder): TransformerEncoder(
    (blocks): ModuleList(
      (0-6): 7 x ModuleDict(
        (attention_normalization): LayerNorm((192,), eps=1e-05, elementwise_affine=True)
        (attention): MultiheadAttention(
          (W_q): Linear(in_features=192, out_features=192, bias=True)
          (W_k): Linear(in_features=192, out_features=192, bias=True)
          (W_v): Linear(in_features=192, out_features=192, bias=True)
          (W_out): Linear(in_features=192, out_features=192, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (attention_residual_dropout): Dropout(p=0.1, inplace=False)
        (ffn_normalization): LayerNorm((192,), eps=1e-05, elementwise_affine=True)
  

In [ ]:
BEST_EPOCH = 46  # replace with the epoch number you found
SAVE_DIR   = f'{CKPT_DIR}/masksdm_splot'
CHECKPOINT_PATH = f"{SAVE_DIR}/epoch_{BEST_EPOCH}.pt"

print(f"Checkpoint: {CHECKPOINT_PATH}")

Checkpoint: /content/drive/MyDrive/MaskSDM/model_checkpoints_1339/masksdm_splot/epoch_40.pt


## 5. Inference with 100% Masking (p=1.0)

All tabular predictors are masked — the model receives only the learned mask token for every input.
This matches STEM-LM p=1.0 (fully unconditioned).
MaskSDM is trained with masked data modelling so this is the setting it is designed to handle gracefully.

In [ ]:
model = get_model(config).to(device)
state_dict = torch.load(CHECKPOINT_PATH, map_location=device)
model.load_state_dict(state_dict)
model.eval()
print('Model loaded.')

Model loaded.


In [ ]:
test_loader = DataLoader(
    get_torch_dataset(config, data['x_test'], data['y_test'], data['satclip_embeddings_test']),
    batch_size=config['batch_size_eval'],
    shuffle=False,
)

all_probs, all_targets = [], []

with torch.no_grad():
    for batch in test_loader:
        x_batch, y_batch, satclip_emb = batch
        x_batch     = x_batch.to(device)
        satclip_emb = satclip_emb.to(device)

        # Match upstream MaskSDM evaluate(): keep all valid features (mask=1=kept).
        # SatCLIP stays masked because we did not train with SatCLIP embeddings.
        x_mask       = ~torch.isnan(x_batch).to(device)
        satclip_mask = torch.zeros(len(satclip_emb), dtype=torch.bool, device=device)

        logits = model(x_batch, satclip_emb, x_mask, satclip_mask)
        probs  = torch.sigmoid(logits)

        all_probs.append(probs.cpu().numpy())
        all_targets.append(y_batch.cpu().numpy())

probs   = np.concatenate(all_probs,   axis=0)                    # (N_test, S)
targets = np.concatenate(all_targets, axis=0).astype(np.int64)   # (N_test, S)

print(f'probs:   {probs.shape}')
print(f'targets: {targets.shape}')

probs:   (9769, 1201)
targets: (9769, 1201)


## 6. STEM-LM Metrics

In [ ]:
# ── Identical metric primitives to CISO notebook (STEMLM_metric.py) ──────────

def _safe_auc_roc(y, p):
    if y.size == 0 or len(set(y.tolist())) < 2 or np.isnan(p).any(): return float('nan')
    try: return float(roc_auc_score(y, p))
    except Exception: return float('nan')

def _safe_auc_pr(y, p):
    if y.size == 0 or y.sum() == 0 or y.sum() == y.size or np.isnan(p).any(): return float('nan')
    try: return float(average_precision_score(y, p))
    except Exception: return float('nan')

def _safe_brier(y, p):
    if y.size == 0 or np.isnan(p).any(): return float('nan')
    return float(np.mean((p - y.astype(np.float64))**2))

def _safe_ece(y, p, n_bins=15):
    if y.size == 0 or np.isnan(p).any(): return float('nan')
    edges = np.linspace(0, 1, n_bins+1)
    idx   = np.clip(np.digitize(p, edges) - 1, 0, n_bins-1)
    err = 0.0; n = p.size
    for b in range(n_bins):
        m = idx == b
        if not m.any(): continue
        err += (m.sum()/n) * abs(y[m].mean() - p[m].mean())
    return float(err)

def _safe_cbi(y, p, n_windows=101, width=0.1, min_per_window=10):
    if y.size == 0 or y.sum() == 0 or y.sum() == y.size or np.isnan(p).any(): return float('nan')
    pres = p[y==1]; bg = p[y==0]
    if pres.size == 0 or bg.size == 0: return float('nan')
    centers = np.linspace(0, 1, n_windows); half = width/2
    pe = np.full(n_windows, np.nan)
    for i, c in enumerate(centers):
        lo, hi = c-half, c+half
        n_bg = int(((bg>=lo)&(bg<=hi)).sum())
        if n_bg < min_per_window: continue
        e = n_bg/bg.size
        if e == 0: continue
        pe[i] = ((pres>=lo)&(pres<=hi)).sum()/pres.size / e
    ok = np.isfinite(pe)
    if ok.sum() < 3 or np.unique(pe[ok]).size < 2: return float('nan')
    try:
        rho = spearmanr(centers[ok], pe[ok]).statistic
        return float(rho) if np.isfinite(rho) else float('nan')
    except Exception: return float('nan')

print('Metric functions defined.')

Metric functions defined.


In [ ]:
# At p=1.0 every feature is masked for every row, so we use all test rows.
# Per-split species filter: skip species with no presences (or no absences) in test
# — matches how the R baselines (Logistic / GAM / Maxnet) report n_species.
auc_roc_vals, auc_pr_vals, cbi_vals, brier_vals, ece_vals = [], [], [], [], []

for s in range(targets.shape[1]):
    y = targets[:, s].astype(np.int64)
    p = probs[:, s].astype(np.float64)
    if y.sum() == 0 or y.sum() == len(y):
        continue
    auc_roc_vals.append(_safe_auc_roc(y, p))
    auc_pr_vals.append(_safe_auc_pr(y, p))
    cbi_vals.append(_safe_cbi(y, p))
    brier_vals.append(_safe_brier(y, p))
    ece_vals.append(_safe_ece(y, p))

def _nanmean(v):   return float(np.nanmean([x for x in v if np.isfinite(x)])) if v else float('nan')
def _nanq(v, q):   vals=[x for x in v if np.isfinite(x)]; return float(np.quantile(vals,q)) if vals else float('nan')

summary = {
    'model':          'MaskSDM',
    'masking_p':      1.0,
    'eval_known_ratio': 0.0,
    'n_species':      len(auc_roc_vals),
    'mean_auc_roc':   _nanmean(auc_roc_vals),
    'auc_roc_q25':    _nanq(auc_roc_vals, 0.25),
    'auc_roc_q50':    _nanq(auc_roc_vals, 0.50),
    'auc_roc_q75':    _nanq(auc_roc_vals, 0.75),
    'mean_auc_pr':    _nanmean(auc_pr_vals),
    'mean_cbi':       _nanmean(cbi_vals),
    'mean_brier':     _nanmean(brier_vals),
    'mean_ece':       _nanmean(ece_vals),
}

cols = ['mean_auc_roc','auc_roc_q25','auc_roc_q50','auc_roc_q75',
        'mean_auc_pr','mean_cbi','mean_brier','mean_ece','n_species']

print('\n-- MaskSDM Benchmark (p=1.0, fully unconditioned) --')
for k in cols:
    v = summary[k]
    print(f'  {k:<20} {round(v, 4) if isinstance(v, float) else v}')


-- MaskSDM Benchmark (p=1.0, fully unconditioned) --
  mean_auc_roc         0.9466
  auc_roc_q25          0.923
  auc_roc_q50          0.9569
  auc_roc_q75          0.9811
  mean_auc_pr          0.1877
  mean_cbi             0.6705
  mean_brier           0.0738
  mean_ece             0.1034
  n_species            1050


In [ ]:
import json as _json

Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)

out_csv  = f'{RESULTS_DIR}/masksdm_benchmark_summary.csv'
out_json = f'{RESULTS_DIR}/masksdm_benchmark_summary.json'

pd.DataFrame([summary]).set_index('masking_p').to_csv(out_csv)
with open(out_json, 'w') as f:
    _json.dump(summary, f, indent=2)

print(f'CSV  saved to {out_csv}')
print(f'JSON saved to {out_json}')

CSV  saved to /content/drive/MyDrive/MaskSDM/results_plants_1000_epochs_1339/masksdm_benchmark_summary.csv
JSON saved to /content/drive/MyDrive/MaskSDM/results_plants_1000_epochs_1339/masksdm_benchmark_summary.json
